# 05 – Scenario Attribution

Isolates how much each dynamic input variable contributed to the change in
predicted nitrogen load between the **early period (2001–2005)** and the
**late period (2016–2020)**.

**Approach — counterfactual substitution:**
For each scenario variable (or group), the 2016–2020 values are replaced by
their corresponding early-period equivalents (year mapping: 2016→2001, …,
2020→2005), and the trained model is re-run.  Comparing the scenario
prediction to the baseline (unmodified 2016–2020) prediction reveals how
much that variable's change drove the load trend.

Variables tested individually and as groups:

| Category | Variables |
|---|---|
| Land management | `cover_crop_percent`, `wetlands_percent`, `forest_percent`, `CRP_percent`, `no_till` |
| Climate | `PPT`, `meanTemp` |
| Hydrology | `meanq`, `strmloss`, `iresload` |

**Prerequisite:** trained checkpoint from notebook 02.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

from sparrow import SPARROW, ParamGenerator, hydseq, setup_seed

setup_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Paths (edit these to point to your data) ─────────────────────────────────
# sparrow_input_path : same file as in notebook 01 (see column list there)
# trendn_path        : same TREND-N file as in notebook 01 (see column list there)
sparrow_input_path = 'data/sparrow_input.csv'               # <-- CHANGE THIS
trendn_path        = 'data/trendn_surplus.csv'              # <-- CHANGE THIS
checkpoint_path    = 'outputs/checkpoint_epoch_150.pth'   # trained checkpoint
output_dir         = 'outputs/scenario_attribution/'
os.makedirs(output_dir, exist_ok=True)


## Load and prepare baseline data

In [ ]:
input_data_huc12 = pd.read_csv(sparrow_input_path)
input_data_huc12['deliver_0'] = 0

surplus = pd.read_csv(trendn_path)
surplus['waterid'] = surplus['waterid'].astype(int)
input_data_huc12 = input_data_huc12.merge(surplus, on='waterid', how='left')
input_data_huc12 = input_data_huc12.fillna(0)

# ── N budget → kg/yr ──────────────────────────────────────────────────────────
input_data_huc12['area_ha'] = input_data_huc12['demiarea'] * 100
cols_inputs  = ['Atmospheric_Oxidized', 'Atmospheric_Reduced',
                'Fertilizer_Agriculture', 'Fertilizer_NonAgriculture',
                'Fix_Cropland', 'Fix_Pasture', 'Human',
                'Lvst_BeefCow', 'Lvst_Broilers', 'Lvst_DairyCow', 'Lvst_Equine',
                'Lvst_Hogs', 'Lvst_LayersPullets', 'Lvst_OtherCattle',
                'Lvst_SheepGoat', 'Lvst_Turkeys']
cols_uptakes = ['CropUptake_Cropland', 'CropUptake_Pasture']
input_data_huc12['N_surplus'] = (input_data_huc12[cols_inputs].sum(axis=1)
                                  - input_data_huc12[cols_uptakes].sum(axis=1))
input_data_huc12['total_N_surplus'] = input_data_huc12['N_surplus'] * input_data_huc12['area_ha']

# ── Node remapping ────────────────────────────────────────────────────────────
input_data_huc12['new_waterid'] = range(1, len(input_data_huc12) + 1)
all_nodes    = pd.unique(input_data_huc12[['fnode', 'tnode']].values.ravel('K'))
node_mapping = {n: i + 1 for i, n in enumerate(all_nodes) if pd.notna(n)}
input_data_huc12['from_node'] = input_data_huc12['fnode'].map(node_mapping)
input_data_huc12['to_node']   = input_data_huc12['tnode'].map(node_mapping)
nan_mask = input_data_huc12['to_node'].isna()
start_id = len(input_data_huc12) + 2
input_data_huc12.loc[nan_mask, 'to_node'] = range(start_id, start_id + nan_mask.sum())

# ── HYDSEQ ────────────────────────────────────────────────────────────────────
df_hseq = hydseq(input_data_huc12[['waterid', 'fnode', 'tnode']])
if 'hydseq' in input_data_huc12.columns:
    input_data_huc12 = input_data_huc12.drop(columns=['hydseq'])
input_data_huc12 = input_data_huc12.merge(
    df_hseq[['seqvar', 'hydseq']], left_index=True, right_on='seqvar', how='left')

# ── HUC_12 identifier (waterid without 2-digit year suffix) ──────────────────
input_data_huc12['HUC_12'] = input_data_huc12['waterid'].astype(str).str[:-2]
print(f"Loaded {len(input_data_huc12):,} reach-year rows.")


## Fit scalers and load trained checkpoint

In [ ]:
source_columns    = ['total_N_surplus']
delivery_columns  = ['deliver_0']
stream_columns    = ['strmloss']
reservoir_columns = ['iresload']

input_columns      = ['PPT', 'tiles_perc', 'soil_CLAYAVE', 'meanTemp',
                       'CRP_percent', 'no_till', 'cover_crop_percent',
                       'forest_percent', 'wetlands_percent']
input_columns_strm = ['slope', 'meanq']
input_columns_res  = ['meanTemp']

# Mean-centre delivery variable
input_data_huc12['deliver_0'] -= input_data_huc12['deliver_0'].mean()

# Scalers fit on all-year baseline data
scaler      = MinMaxScaler().fit(input_data_huc12[input_columns])
scaler_strm = MinMaxScaler().fit(input_data_huc12[input_columns_strm])
scaler_res  = MinMaxScaler().fit(input_data_huc12[input_columns_res])

dlvdsgn = torch.zeros(len(source_columns), len(delivery_columns), dtype=torch.float32, device=device)

ns = len(source_columns); nd = len(delivery_columns)

param_model = ParamGenerator(
    input_size=len(input_columns), hidden_size=32,
    num_source=ns, num_delivery=nd,
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns)).to(device)
param_model_strm = ParamGenerator(
    input_size=len(input_columns_strm), hidden_size=8,
    num_source=ns, num_delivery=nd,
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns)).to(device)
param_model_res = ParamGenerator(
    input_size=len(input_columns_res), hidden_size=8,
    num_source=ns, num_delivery=nd,
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns)).to(device)

ckpt = torch.load(checkpoint_path, map_location=device)
param_model.load_state_dict(ckpt['model_state_dict'])
param_model_strm.load_state_dict(ckpt['param_model_strm_state_dict'])
param_model_res.load_state_dict(ckpt['param_model_res_state_dict'])
param_model.eval(); param_model_strm.eval(); param_model_res.eval()

sparrow_model = SPARROW().to(device)
print('Checkpoint loaded.')


## Define early-period substitution

For the late period (2016–2020), we prepare a look-up of early-period
(2001–2005) values matched by `HUC_12` and the shifted year.  Each scenario
replaces one variable (or a group) in the late period with its early-period
counterpart and re-runs the model.


In [ ]:
# Year mapping: late-period year → corresponding early-period year
year_map = {2016: 2001, 2017: 2002, 2018: 2003, 2019: 2004, 2020: 2005}

# All variables that can be substituted
sub_vars = ['cover_crop_percent', 'wetlands_percent', 'forest_percent',
            'CRP_percent', 'no_till', 'PPT', 'meanTemp',
            'meanq', 'strmloss', 'iresload']

# Build the early-period lookup (2001-2005 rows, tagged with their late-period year)
df_0105 = input_data_huc12[input_data_huc12['Year'].isin([2001, 2002, 2003, 2004, 2005])].copy()
df_0105['Year_sub'] = df_0105['Year'].map({2001: 2016, 2002: 2017, 2003: 2018,
                                            2004: 2019, 2005: 2020})
df_merge = input_data_huc12.merge(
    df_0105[['HUC_12', 'Year_sub'] + sub_vars],
    left_on=['HUC_12', 'Year'],
    right_on=['HUC_12', 'Year_sub'],
    how='left',
    suffixes=('', '_from0105'))

mask_1620 = input_data_huc12['Year'].between(2016, 2020)

# Scenario groups: name → list of variables to revert to early-period values
scenarios = {
    'baseline':     [],
    'cover_crop':   ['cover_crop_percent'],
    'wetlands':     ['wetlands_percent'],
    'forest':       ['forest_percent'],
    'CRP':          ['CRP_percent'],
    'no_till':      ['no_till'],
    'PPT':    ['PPT'],
    'meanTemp':     ['meanTemp'],
    'hydro':    ['PPT', 'meanTemp'],
    'human':     ['cover_crop_percent', 'wetlands_percent', 'forest_percent',
                     'CRP_percent', 'no_till'],
    'all_vars':     sub_vars,
}
print(f"Defined {len(scenarios)} scenarios.")


## Run all scenarios

For each scenario, the late-period (2016–2020) rows are modified, the model
is run in eval mode, and predictions for all reaches are saved to a CSV.


In [ ]:
def run_sparrow_for_years(df, target_years):
    """Run trained model on target_years rows of df; return DataFrame of predictions."""
    records = []
    with torch.no_grad():
        for year in target_years:
            yd = df[df['Year'] == year].copy()

            def _t(col, dtype=torch.float32):
                return torch.tensor(yd[col].values, dtype=dtype, device=device)

            X_sc   = torch.tensor(scaler.transform(yd[input_columns]),
                                  dtype=torch.float32, device=device)
            X_strm = torch.tensor(scaler_strm.transform(yd[input_columns_strm]),
                                  dtype=torch.float32, device=device)
            X_res  = torch.tensor(scaler_res.transform(yd[input_columns_res]),
                                  dtype=torch.float32, device=device)

            coeffs      = param_model(X_sc)
            coeffs_strm = param_model_strm(X_strm)
            coeffs_res  = param_model_res(X_res)

            alpha   = coeffs[:, :ns]
            theta_D = coeffs[:, ns:ns + nd]
            theta_S = coeffs_strm[:, -2:-1]
            theta_R = coeffs_res[:, -1:]

            _, _, _, total_load, orig_cid = sparrow_model(
                _t(source_columns), _t(delivery_columns),
                _t(stream_columns), _t(reservoir_columns),
                _t('rchtype', torch.int64), _t('hydseq'),
                _t('headflag', torch.int64),
                _t('from_node', torch.int64), _t('to_node', torch.int64),
                alpha, theta_D, theta_S, theta_R,
                _t('frac'), _t('new_waterid', torch.int64),
                _t('waterid', torch.int64), dlvdsgn,
                _t('iftran', torch.int64))

            records.append(pd.DataFrame({
                'waterid': orig_cid.cpu().numpy(),
                'Year': year,
                'total_load': total_load.cpu().numpy(),
            }))
    return pd.concat(records, ignore_index=True)


target_years = [2016, 2017, 2018, 2019, 2020]

for scenario_name, vars_to_sub in scenarios.items():
    # Work on a copy so baseline is never modified
    df_scenario = input_data_huc12.copy()

    # Substitute late-period values with early-period equivalents
    for var in vars_to_sub:
        df_scenario.loc[mask_1620, var] = df_merge.loc[mask_1620, f'{var}_from0105'].values

    df_pred = run_sparrow_for_years(df_scenario, target_years)

    out_path = os.path.join(output_dir, f'scenario_{scenario_name}.csv')
    df_pred.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({len(df_pred):,} rows)")

print(f"\nAll {len(scenarios)} scenarios complete.")


## Compare scenarios to baseline

In [ ]:
import glob

baseline = pd.read_csv(os.path.join(output_dir, 'scenario_baseline.csv'))
baseline = baseline.rename(columns={'total_load': 'load_baseline'})

rows = []
for scenario_name in scenarios:
    if scenario_name == 'baseline':
        continue
    sc = pd.read_csv(os.path.join(output_dir, f'scenario_{scenario_name}.csv'))
    sc = sc.rename(columns={'total_load': 'load_scenario'})
    merged = baseline.merge(sc, on=['waterid', 'Year'])
    merged['load_diff'] = merged['load_scenario'] - merged['load_baseline']
    basin_diff = merged['load_diff'].sum()
    basin_base = merged['load_baseline'].sum()
    rows.append({'scenario': scenario_name,
                 'basin_load_diff_kg': basin_diff,
                 'pct_change': basin_diff / basin_base * 100})

summary = pd.DataFrame(rows).sort_values('basin_load_diff_kg')
print(summary.to_string(index=False))
summary.to_csv(os.path.join(output_dir, 'scenario_summary.csv'), index=False)
